In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)


FileUpload(value=(), accept='*', description='Upload')

In [5]:
print(uploader.value)

({'name': '00d4468119e12b029a072aa4fef09749.jpg', 'type': 'image/jpeg', 'size': 1169519, 'content': <memory at 0x00000246897C1480>, 'last_modified': datetime.datetime(2026, 9, 17, 17, 44, 44, 706000, tzinfo=datetime.timezone.utc)},)


In [6]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件

uploaded_file = list(uploader.value)[0]
# 获取其内存视图
content_mv = uploaded_file["content"]
# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()
# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")


In [7]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

model = init_chat_model(
    model="deepseek-chat",
    model_provider="openai",
    base_url="https://api.deepseek.com",
    api_key=api_key
)


In [8]:
# 组织多模态消息
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])
from langchain.agents import create_agent
agent = create_agent(model)
for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

这张图片中其实**并没有出现任何城市**。

图片的背景是非常纯粹的、被虚化的暖色调光影（呈现出夕阳或日出时的橙黄色光晕），没有建筑、街道或任何城市景观。

这张图的主体是一位极具个性的女性人物插画。画面的视觉焦点完全集中在她的身上：
*   **面部特征**：她有着精致的五官，眼神略带慵懒地看着镜头，眼角有泪痣，涂着红唇。
*   **发型与配饰**：深色的长发被风吹动，戴着显眼的银色大圆环耳环。
*   **穿搭与细节**：她穿着黑色的皮夹克，手托着下巴，脖子处露出了大面积的黑色纹身，右下角还能看到金属质感的机械手臂或装备局部。
*   **光影**：强烈的逆光/侧光打在她的头发和脸颊上，营造出一种唯美又带着点赛博朋克或废土风格的浪漫氛围。

如果您是想了解这幅画作的风格，它属于典型的**二次元半厚涂数字插画**，结合了现代潮流（皮衣、纹身、金属元素）与唯美面容，背景的落日余晖只是为了烘托人物的情绪和轮廓。